In [ ]:
pip install ollama

In [ ]:
pip install langchain langchain-openai pydantic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 31.6 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1


In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (14.2 MB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122363 files and directories currently 

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
%%bash --bg
ollama serve

In [ ]:
!mkdir -p /content/drive/MyDrive/ollama_models
!rm -rf ~/.ollama/models
!mkdir -p ~/.ollama
!ln -s /content/drive/MyDrive/ollama_models ~/.ollama/models

In [ ]:
!rm -rf ~/.ollama/models
!mkdir -p /content/ollama_models
!ln -s /content/ollama_models ~/.ollama/models


In [ ]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.9 MB/s eta 0:00:00


In [ ]:
pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=7a72fff3685a76600511caf0f4de40db8662ff838a59eecf207b5bf96bc10a1a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [ ]:
import subprocess
import time

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
!ollama pull mistral-nemo:12b
!ollama pull qwen2:7b
!ollama pull llama3.1:8b
!ollama pull gemma2:9b
!ollama pull deepseek-r1:8b
!ollama pull phi3:3.8b


In [ ]:
!ollama list

NAME           ID              SIZE     MODIFIED      
gpt-oss:20b    17052f91a42e    13 GB    3 minutes ago    


In [ ]:
import ollama
import json
import time
import math
import csv
import os
import re
from collections import Counter
from datetime import datetime



In [ ]:


# =========================================================
# CONFIGURATION
# =========================================================

TEAM = [
    "qwen2:7b",
    "mistral-nemo:12b",
    "llama3.1:8b",
    "deepseek-r1:8b",
    "phi3:3.8b"
]

# Captain model is NOT part of debate team
CAPTAIN_MODEL = "gemma2:9b"

LOG_FILE = "adaptive_experiment.jsonl"

# Debate trigger thresholds
ENTROPY_THRESHOLD = 0.5
AGREEMENT_THRESHOLD = 0.5

SYSTEM_PROMPT = """
Answer concisely.
Only provide the final answer.
"""

DEBATE_PROMPT = """
You are participating in a collaborative reasoning discussion.

Question:
{question}

Your previous answer:
{previous_answer}

Other model answers:
{other_answers}

Carefully reconsider the question.

If another answer appears more correct, revise your answer.

Only provide your final answer.
"""

CAPTAIN_PROMPT = """
You are the captain model responsible for selecting the best answer.

Question:
{question}

Candidate answers after debate:
{answers}

Select ONLY one answer from the candidate answers above.
Do NOT create a new answer.

Only provide the final selected answer.
"""

# =========================================================
# TEXT NORMALIZATION
# =========================================================

def normalize(text):

    if text is None:
        return ""

    text = text.lower().strip()

    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove common prefixes
    prefixes = [
        "the answer is",
        "answer:",
        "it is",
    ]

    for p in prefixes:
        if text.startswith(p):
            text = text[len(p):].strip()

    return text


# =========================================================
# FILE HANDLING
# =========================================================

def load_existing():

    results = {}

    if not os.path.exists(LOG_FILE):
        return results

    with open(LOG_FILE, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            try:
                entry = json.loads(line)

                if "question_id" in entry:
                    results[entry["question_id"]] = entry

            except Exception:
                continue

    return results


def save_entry(entry):

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")


def get_last_completed_index():

    existing = load_existing()

    if not existing:
        return 0

    return max(existing.keys()) + 1


# =========================================================
# MODEL CALL
# =========================================================

def ask_model(model, prompt, temperature=0.2):

    start = time.time()

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        options={
            "temperature": temperature
        }
    )

    latency = time.time() - start

    answer = response["message"]["content"].strip()

    return {
        "model": model,
        "answer_raw": answer,
        "answer_norm": normalize(answer),
        "latency": latency
    }


# =========================================================
# INITIAL ANSWERS
# =========================================================

def generate_initial_answers(question):

    answers = []

    for model in TEAM:

        print(f"  → Initial: {model}")

        result = ask_model(
            model=model,
            prompt=question
        )

        answers.append(result)

    return answers


# =========================================================
# DISAGREEMENT ANALYSIS
# =========================================================

def compute_disagreement(answers):

    votes = Counter(
        a["answer_norm"]
        for a in answers
    )

    total_models = len(answers)

    unique_answers = len(votes)

    max_votes = max(votes.values())

    agreement_rate = max_votes / total_models

    entropy = -sum(
        (count / total_models)
        * math.log2(count / total_models)
        for count in votes.values()
    )

    # Disagreement category
    if entropy < 0.3:
        disagreement_level = "low"

    elif entropy < 1.0:
        disagreement_level = "medium"

    else:
        disagreement_level = "high"

    return {
        "votes": dict(votes),
        "unique_answers": unique_answers,
        "max_votes": max_votes,
        "agreement_rate": agreement_rate,
        "entropy": entropy,
        "disagreement_level": disagreement_level
    }


# =========================================================
# DEBATE PHASE
# =========================================================

def run_debate(question, initial_answers):

    debated_answers = []

    for agent in initial_answers:

        model = agent["model"]

        other_answers = []

        for other in initial_answers:

            if other["model"] != model:

                other_answers.append(
                    f"{other['model']}: {other['answer_raw']}"
                )

        prompt = DEBATE_PROMPT.format(
            question=question,
            previous_answer=agent["answer_raw"],
            other_answers="\n".join(other_answers)
        )

        print(f"  → Debate: {model}")

        debated = ask_model(
            model=model,
            prompt=prompt
        )

        debated_answers.append(debated)

    return debated_answers


# =========================================================
# CAPTAIN ARBITRATION
# =========================================================

def captain_select(question, debated_answers):

    formatted_answers = "\n".join(
        [
            f"{a['model']}: {a['answer_raw']}"
            for a in debated_answers
        ]
    )

    prompt = CAPTAIN_PROMPT.format(
        question=question,
        answers=formatted_answers
    )

    print(f"  → Captain selecting final answer")

    captain_result = ask_model(
        model=CAPTAIN_MODEL,
        prompt=prompt
    )

    return captain_result["answer_norm"]


# =========================================================
# ADAPTIVE COORDINATION
# =========================================================

def run_adaptive_question(
    question,
    true_answer=None,
    difficulty=None
):

    # -----------------------------------------------------
    # STEP 1: INITIAL RESPONSES
    # -----------------------------------------------------

    initial_answers = generate_initial_answers(question)

    disagreement = compute_disagreement(initial_answers)

    entropy = disagreement["entropy"]
    agreement_rate = disagreement["agreement_rate"]

    # -----------------------------------------------------
    # STEP 2: DECIDE WHETHER TO DEBATE
    # -----------------------------------------------------

    debate_triggered = (
        entropy > ENTROPY_THRESHOLD
        or agreement_rate < AGREEMENT_THRESHOLD
    )

    # =====================================================
    # LOW DISAGREEMENT
    # =====================================================

    if not debate_triggered:

        print("  ✓ Low disagreement → Majority voting")

        votes = Counter(
            a["answer_norm"]
            for a in initial_answers
        )

        final_answer = votes.most_common(1)[0][0]

        debated_answers = None
        post_debate_disagreement = None

        strategy_used = "majority_vote"

        total_latency = sum(
            a["latency"]
            for a in initial_answers
        )

    # =====================================================
    # HIGH DISAGREEMENT
    # =====================================================

    else:

        print("  ⚠ High disagreement → Debate triggered")

        debated_answers = run_debate(
            question,
            initial_answers
        )

        # -------------------------------------------------
        # POST-DEBATE ANALYSIS
        # -------------------------------------------------

        post_debate_disagreement = compute_disagreement(
            debated_answers
        )

        debate_votes = Counter(
            a["answer_norm"]
            for a in debated_answers
        )

        most_common = debate_votes.most_common()

        top_count = most_common[0][1]

        tied_answers = [
            ans
            for ans, count in most_common
            if count == top_count
        ]

        # -------------------------------------------------
        # CLEAR MAJORITY AFTER DEBATE
        # -------------------------------------------------

        if len(tied_answers) == 1:

            final_answer = tied_answers[0]

            strategy_used = "debate_majority"

            print("  ✓ Debate converged → Majority selected")

        # -------------------------------------------------
        # TIE AFTER DEBATE
        # -------------------------------------------------

        else:

            print("  ⚠ Debate tie → Captain arbitration")

            final_answer = captain_select(
                question,
                debated_answers
            )

            strategy_used = "debate_captain"

        total_latency = (
            sum(a["latency"] for a in initial_answers)
            + sum(a["latency"] for a in debated_answers)
        )

    # =====================================================
    # FINAL RESULT ENTRY
    # =====================================================

    entry = {

        # Metadata
        "timestamp": datetime.utcnow().isoformat(),
        "question": question,
        "true_answer": true_answer,
        "difficulty": difficulty,

        # Strategy
        "strategy_used": strategy_used,
        "debate_triggered": debate_triggered,

        # Initial disagreement
        "entropy": disagreement["entropy"],
        "agreement_rate": disagreement["agreement_rate"],
        "unique_answers": disagreement["unique_answers"],
        "max_votes": disagreement["max_votes"],
        "disagreement_level": disagreement["disagreement_level"],
        "vote_distribution": disagreement["votes"],

        # Post debate disagreement
        "post_debate_disagreement": post_debate_disagreement,

        # Latency
        "total_latency": total_latency,

        # Answers
        "initial_answers": initial_answers,
        "debated_answers": debated_answers,
        "final_answer": final_answer
    }

    # =====================================================
    # CORRECTNESS
    # =====================================================

    if true_answer:

        entry["correct"] = (
            normalize(true_answer)
            == normalize(final_answer)
        )

    return entry


# =========================================================
# FULL EXPERIMENT
# =========================================================

def run_experiment(
    csv_path,
    start_index=None,
    limit=None,
    question_column="question",
    answer_column="answer",
    difficulty_column="difficulty"
):

    existing = load_existing()

    if start_index is None:
        start_index = get_last_completed_index()

    with open(
        csv_path,
        newline="",
        encoding="utf-8"
    ) as f:

        rows = list(csv.DictReader(f))

    total = len(rows)

    end_index = (
        total
        if limit is None
        else min(start_index + limit, total)
    )

    for i in range(start_index, end_index):

        if i in existing:

            print(f"Skipping Q{i} (already completed)")

            continue

        row = rows[i]

        question = row[question_column]

        true_answer = (
            row[answer_column]
            if answer_column and answer_column in row
            else None
        )

        difficulty = (
            row[difficulty_column]
            if difficulty_column in row
            else None
        )

        print("\n" + "=" * 70)
        print(f"QUESTION {i}")
        print("=" * 70)
        print(question)

        try:

            entry = run_adaptive_question(
                question=question,
                true_answer=true_answer,
                difficulty=difficulty
            )

            entry["question_id"] = i

            save_entry(entry)

        except Exception as e:

            print(f"ERROR on question {i}: {e}")

            error_entry = {
                "timestamp": datetime.utcnow().isoformat(),
                "question_id": i,
                "question": question,
                "error": str(e)
            }

            save_entry(error_entry)

    print("\nExperiment completed.")

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
# =========================================================
# RUN
# =========================================================

run_experiment(
    csv_path="trivia_questions (3).csv",
    start_index=0,
    limit=1000,
    question_column="question",
    answer_column="answer",
    difficulty_column="difficulty"
)


QUESTION 0
Spoiler alert! In "Return of the Jedi", Luke sets this man ablaze in a funeral pyre
  → Initial: qwen2:7b
  → Initial: mistral-nemo:12b
  → Initial: llama3.1:8b
  → Initial: deepseek-r1:8b
  → Initial: phi3:3.8b
  ⚠ High disagreement → Debate triggered
  → Debate: qwen2:7b
  → Debate: mistral-nemo:12b
  → Debate: llama3.1:8b
  → Debate: deepseek-r1:8b
  → Debate: phi3:3.8b
  ✓ Debate converged → Majority selected

QUESTION 1
The FAA lists airports as small, medium or large within this 3-letter designation that means you change planes there
  → Initial: qwen2:7b


/tmp/ipykernel_608/1137063369.py:442: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),


  → Initial: mistral-nemo:12b
  → Initial: llama3.1:8b
  → Initial: deepseek-r1:8b
  → Initial: phi3:3.8b
  ⚠ High disagreement → Debate triggered
  → Debate: qwen2:7b
  → Debate: mistral-nemo:12b
  → Debate: llama3.1:8b
  → Debate: deepseek-r1:8b
  → Debate: phi3:3.8b
  ⚠ Debate tie → Captain arbitration
  → Captain selecting final answer

QUESTION 2
Wing Commander III features this Luke Skywalker actor as the wing commander
  → Initial: qwen2:7b
  → Initial: mistral-nemo:12b
  → Initial: llama3.1:8b
  → Initial: deepseek-r1:8b
  → Initial: phi3:3.8b
  ⚠ High disagreement → Debate triggered
  → Debate: qwen2:7b
  → Debate: mistral-nemo:12b
  → Debate: llama3.1:8b
  → Debate: deepseek-r1:8b
  → Debate: phi3:3.8b
  ✓ Debate converged → Majority selected

QUESTION 3
Betty Crocker sells this snack food under the name Pop Secret
  → Initial: qwen2:7b
  → Initial: mistral-nemo:12b
  → Initial: llama3.1:8b
  → Initial: deepseek-r1:8b
  → Initial: phi3:3.8b
  ⚠ High disagreement → Debate trig

KeyboardInterrupt: 

In [ ]:
import pandas as pd


In [ ]:
LOG_FILE= "/content/adaptive_entropy_experiment (2) (7).jsonl"

In [ ]:


data = []

with open(LOG_FILE, "r") as f:
    for line in f:
        data.append(json.loads(line))

df = pd.DataFrame(data)

NameError: name 'json' is not defined

In [ ]:
df

,timestamp,question,true_answer,difficulty,strategy_used,debate_triggered,entropy,agreement_rate,unique_answers,max_votes,disagreement_level,vote_distribution,total_latency,initial_answers,debated_answers,final_answer,correct,question_id
0,2026-05-12T09:54:09.591827,"Spoiler alert! In ""Return of the Jedi"", Luke s...",Darth Vader,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'darth vader': 2, 'vader': 1, 'admiral motti'...",48.411473,"[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...","[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...",darth vader,True,0
1,2026-05-12T09:55:50.766390,Betty Crocker sells this snack food under the ...,Microwave popcorn,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'pop secret': 2, 'popcorn': 1, 'no': 1, 'no b...",28.448211,"[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...","[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...",pop secret,False,3
2,2026-05-12T09:56:28.532776,"We're told this ""doesn't grow on trees"", but i...",money,easy,debate_captain,True,2.321928,0.2,5,1,high,"{'candle': 1, 'money': 1, 'a wedding ring': 1,...",37.479811,"[{'model': 'qwen2:7b', 'answer_raw': 'Candle',...","[{'model': 'qwen2:7b', 'answer_raw': 'wedding ...",wedding ring,False,4
3,2026-05-12T09:56:57.741372,Belle Boyd worked as a spy for Stonewall Jacks...,Shenandoah,easy,debate_captain,True,1.370951,0.6,3,3,high,"{'shenandoah valley': 3, 'shenandoah': 1, 'fal...",28.922996,"[{'model': 'qwen2:7b', 'answer_raw': 'Shenando...","[{'model': 'qwen2:7b', 'answer_raw': 'Shenando...",shenandoah valley,False,5
4,2026-05-12T09:57:27.587463,This sandwich named for its meat & dairy filli...,Philly Cheesesteak,easy,debate_captain,True,0.721928,0.8,2,4,medium,"{'cheesesteak': 4, 'hoagie': 1}",29.568331,"[{'model': 'qwen2:7b', 'answer_raw': 'Cheesest...","[{'model': 'qwen2:7b', 'answer_raw': 'Cheesest...",cheesesteak,False,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,2026-05-13T15:44:48.881470,"Founded in 1592, it's also called the Universi...",Trinity College,medium,debate_captain,True,0.721928,0.8,2,4,medium,"{'trinity college dublin': 4, 'trinity college...",27.511527,"[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...","[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...",trinity college dublin,False,997
996,2026-05-13T15:45:18.250502,"Seen here, it's located in the Hôtel des Inval...",Napoleon\'s Tomb,medium,debate_captain,True,2.321928,0.2,5,1,high,{'the final answer the tomb of napoleon bonapa...,29.046867,"[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...","[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...",the tomb of napoleon bonaparte,False,998
997,2026-05-13T15:45:48.266105,"On April 6, 1917 America entered World War I &...",George M. Cohan,medium,debate_captain,True,1.370951,0.6,3,3,high,"{'jules jolivet': 1, 'george m cohan': 3, 'edw...",29.699222,"[{'model': 'qwen2:7b', 'answer_raw': 'Jules Jo...","[{'model': 'qwen2:7b', 'answer_raw': 'George M...",george m cohan,True,999
998,2026-05-13T16:16:03.159216,"The FAA lists airports as small, medium or lar...",hub,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'hub': 2, 'medium': 1, 'the faa classifies ai...",49.700103,"[{'model': 'qwen2:7b', 'answer_raw': 'HUB', 'a...","[{'model': 'qwen2:7b', 'answer_raw': 'HUB', 'a...",hub,True,1


In [ ]:
final_df.to_csv('adaptive_entropy_experiment.csv')

In [ ]:
df['question']

,question
0,"Spoiler alert! In ""Return of the Jedi"", Luke s..."
1,Betty Crocker sells this snack food under the ...
2,"We're told this ""doesn't grow on trees"", but i..."
3,Belle Boyd worked as a spy for Stonewall Jacks...
4,This sandwich named for its meat & dairy filli...
...,...
995,"Founded in 1592, it's also called the Universi..."
996,"Seen here, it's located in the Hôtel des Inval..."
997,"On April 6, 1917 America entered World War I &..."
998,"The FAA lists airports as small, medium or lar..."


In [ ]:
questions['question']

,question
0,"Spoiler alert! In ""Return of the Jedi"", Luke s..."
1,"The FAA lists airports as small, medium or lar..."
2,Wing Commander III features this Luke Skywalke...
3,Betty Crocker sells this snack food under the ...
4,"We're told this ""doesn't grow on trees"", but i..."
...,...
995,Airports have been slow to welcome rideshare c...
996,Poor bone marrow function is the leading cause...
997,"Founded in 1592, it's also called the Universi..."
998,"Seen here, it's located in the Hôtel des Inval..."


In [ ]:
final_df = questions.merge(
    df,
    on="question",
    how="left"
)

In [ ]:
sum(final_df['question']==questions['question'])

1000

In [ ]:
missing = final_df[final_df['answer'].isna()]

print(len(missing))

0


In [ ]:
!ollama pull gpt-oss:20b

In [ ]:
!ollama rm gemma2:9b
!ollama rm deepseek-r1:8b

In [ ]:
df

,timestamp,question,true_answer,difficulty,strategy_used,debate_triggered,entropy,agreement_rate,unique_answers,max_votes,disagreement_level,vote_distribution,total_latency,initial_answers,debated_answers,final_answer,correct,question_id
0,2026-05-12T09:54:09.591827,"Spoiler alert! In ""Return of the Jedi"", Luke s...",Darth Vader,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'darth vader': 2, 'vader': 1, 'admiral motti'...",48.411473,"[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...","[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...",darth vader,True,0
1,2026-05-12T09:55:50.766390,Betty Crocker sells this snack food under the ...,Microwave popcorn,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'pop secret': 2, 'popcorn': 1, 'no': 1, 'no b...",28.448211,"[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...","[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...",pop secret,False,3
2,2026-05-12T09:56:28.532776,"We're told this ""doesn't grow on trees"", but i...",money,easy,debate_captain,True,2.321928,0.2,5,1,high,"{'candle': 1, 'money': 1, 'a wedding ring': 1,...",37.479811,"[{'model': 'qwen2:7b', 'answer_raw': 'Candle',...","[{'model': 'qwen2:7b', 'answer_raw': 'wedding ...",wedding ring,False,4
3,2026-05-12T09:56:57.741372,Belle Boyd worked as a spy for Stonewall Jacks...,Shenandoah,easy,debate_captain,True,1.370951,0.6,3,3,high,"{'shenandoah valley': 3, 'shenandoah': 1, 'fal...",28.922996,"[{'model': 'qwen2:7b', 'answer_raw': 'Shenando...","[{'model': 'qwen2:7b', 'answer_raw': 'Shenando...",shenandoah valley,False,5
4,2026-05-12T09:57:27.587463,This sandwich named for its meat & dairy filli...,Philly Cheesesteak,easy,debate_captain,True,0.721928,0.8,2,4,medium,"{'cheesesteak': 4, 'hoagie': 1}",29.568331,"[{'model': 'qwen2:7b', 'answer_raw': 'Cheesest...","[{'model': 'qwen2:7b', 'answer_raw': 'Cheesest...",cheesesteak,False,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
993,2026-05-13T15:43:50.826633,Airports have been slow to welcome rideshare c...,Atlanta,medium,debate_captain,True,1.921928,0.4,4,2,high,"{'denver international airport': 1, 'jan 1 201...",36.079641,"[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...","[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...",denver international airport,False,995
994,2026-05-13T15:44:21.074118,Poor bone marrow function is the leading cause...,Anemia,medium,debate_captain,True,1.370951,0.6,3,3,high,"{'anemia': 3, 'aplastic anemia': 1, 'diamondbl...",29.960462,"[{'model': 'qwen2:7b', 'answer_raw': 'Anemia',...","[{'model': 'qwen2:7b', 'answer_raw': 'Aplastic...",aplastic anemia,False,996
995,2026-05-13T15:44:48.881470,"Founded in 1592, it's also called the Universi...",Trinity College,medium,debate_captain,True,0.721928,0.8,2,4,medium,"{'trinity college dublin': 4, 'trinity college...",27.511527,"[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...","[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...",trinity college dublin,False,997
996,2026-05-13T15:45:18.250502,"Seen here, it's located in the Hôtel des Inval...",Napoleon\'s Tomb,medium,debate_captain,True,2.321928,0.2,5,1,high,{'the final answer the tomb of napoleon bonapa...,29.046867,"[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...","[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...",the tomb of napoleon bonaparte,False,998


In [ ]:
questions=pd.read_csv('/content/trivia_questions (3).csv')

In [ ]:
questions

,Unnamed: 0.1,Unnamed: 0,ep_num,air_date,round_name,coord,category,value,daily_double,question,answer,correct_attempts,wrong_attempts,value_tuple,value_num,difficulty,topic_cluster,topic_name,human_score
0,0,49222,7555,2017-06-16,Double Jeopardy,"(1, 1)",YOU'RE FIRED FROM THE MOVIE!,"(400,)",False,"Spoiler alert! In ""Return of the Jedi"", Luke s...",Darth Vader,1.0,1.0,"(400,)",400.0,easy,13,Trivia & Famous People,0
1,1,47825,7532,2017-05-16,Jeopardy,"(5, 2)",INFRASTRUCTURE,"(400,)",False,"The FAA lists airports as small, medium or lar...",hub,0.0,3.0,"(400,)",400.0,easy,5,"Mythology, Literature & Wordplay",0
2,2,10333,2742,1996-07-02,Jeopardy,"(3, 3)",STARS ON CD-ROM,"(300,)",False,Wing Commander III features this Luke Skywalke...,Mark Hamill,1.0,0.0,"(300,)",300.0,easy,12,History & World Leaders,1
3,3,20355,2938,1997-05-14,Jeopardy,"(5, 2)",OUT OF THE MICROWAVE,"(200,)",False,Betty Crocker sells this snack food under the ...,Microwave popcorn,1.0,0.0,"(200,)",200.0,easy,12,History & World Leaders,1
4,4,6457,2606,1995-12-25,Jeopardy,"(2, 1)",ETIQUETTE,"(100,)",False,"We're told this ""doesn't grow on trees"", but i...",money,1.0,0.0,"(100,)",100.0,easy,12,History & World Leaders,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,46580,7511,2017-04-17,Jeopardy,"(5, 5)",GET A LYFT WITH UBER,"(1000,)",False,Airports have been slow to welcome rideshare c...,Atlanta,1.0,1.0,"(1000,)",1000.0,medium,12,History & World Leaders,0
996,996,18788,2910,1997-04-04,Double Jeopardy,"(2, 3)",MEDICINE,"(600,)",False,Poor bone marrow function is the leading cause...,Anemia,1.0,0.0,"(600,)",600.0,medium,5,"Mythology, Literature & Wordplay",1
997,997,49635,7562,2017-06-27,Jeopardy,"(2, 5)",OLD SCHOOL,"(1000,)",False,"Founded in 1592, it's also called the Universi...",Trinity College,0.0,3.0,"(1000,)",1000.0,medium,17,Religion,0
998,998,34197,4318,2003-05-14,Jeopardy,"(4, 4)",PARIS SITES,"(800,)",False,"Seen here, it's located in the Hôtel des Inval...",Napoleon\'s Tomb,1.0,0.0,"(800,)",800.0,medium,3,Food & Cuisine,1


In [ ]:
df

,timestamp,question,true_answer,difficulty,strategy_used,debate_triggered,entropy,agreement_rate,unique_answers,max_votes,disagreement_level,vote_distribution,total_latency,initial_answers,debated_answers,final_answer,correct,question_id
0,2026-05-12T09:54:09.591827,"Spoiler alert! In ""Return of the Jedi"", Luke s...",Darth Vader,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'darth vader': 2, 'vader': 1, 'admiral motti'...",48.411473,"[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...","[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...",darth vader,True,0
1,2026-05-12T09:55:50.766390,Betty Crocker sells this snack food under the ...,Microwave popcorn,easy,debate_captain,True,1.921928,0.4,4,2,high,"{'pop secret': 2, 'popcorn': 1, 'no': 1, 'no b...",28.448211,"[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...","[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...",pop secret,False,3
2,2026-05-12T09:56:28.532776,"We're told this ""doesn't grow on trees"", but i...",money,easy,debate_captain,True,2.321928,0.2,5,1,high,"{'candle': 1, 'money': 1, 'a wedding ring': 1,...",37.479811,"[{'model': 'qwen2:7b', 'answer_raw': 'Candle',...","[{'model': 'qwen2:7b', 'answer_raw': 'wedding ...",wedding ring,False,4
3,2026-05-12T09:56:57.741372,Belle Boyd worked as a spy for Stonewall Jacks...,Shenandoah,easy,debate_captain,True,1.370951,0.6,3,3,high,"{'shenandoah valley': 3, 'shenandoah': 1, 'fal...",28.922996,"[{'model': 'qwen2:7b', 'answer_raw': 'Shenando...","[{'model': 'qwen2:7b', 'answer_raw': 'Shenando...",shenandoah valley,False,5
4,2026-05-12T09:57:27.587463,This sandwich named for its meat & dairy filli...,Philly Cheesesteak,easy,debate_captain,True,0.721928,0.8,2,4,medium,"{'cheesesteak': 4, 'hoagie': 1}",29.568331,"[{'model': 'qwen2:7b', 'answer_raw': 'Cheesest...","[{'model': 'qwen2:7b', 'answer_raw': 'Cheesest...",cheesesteak,False,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
993,2026-05-13T15:43:50.826633,Airports have been slow to welcome rideshare c...,Atlanta,medium,debate_captain,True,1.921928,0.4,4,2,high,"{'denver international airport': 1, 'jan 1 201...",36.079641,"[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...","[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...",denver international airport,False,995
994,2026-05-13T15:44:21.074118,Poor bone marrow function is the leading cause...,Anemia,medium,debate_captain,True,1.370951,0.6,3,3,high,"{'anemia': 3, 'aplastic anemia': 1, 'diamondbl...",29.960462,"[{'model': 'qwen2:7b', 'answer_raw': 'Anemia',...","[{'model': 'qwen2:7b', 'answer_raw': 'Aplastic...",aplastic anemia,False,996
995,2026-05-13T15:44:48.881470,"Founded in 1592, it's also called the Universi...",Trinity College,medium,debate_captain,True,0.721928,0.8,2,4,medium,"{'trinity college dublin': 4, 'trinity college...",27.511527,"[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...","[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...",trinity college dublin,False,997
996,2026-05-13T15:45:18.250502,"Seen here, it's located in the Hôtel des Inval...",Napoleon\'s Tomb,medium,debate_captain,True,2.321928,0.2,5,1,high,{'the final answer the tomb of napoleon bonapa...,29.046867,"[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...","[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...",the tomb of napoleon bonaparte,False,998


In [ ]:

missing = questions[~questions['question'].isin(df['question'])]



In [ ]:
missing


,Unnamed: 0.1,Unnamed: 0,ep_num,air_date,round_name,coord,category,value,daily_double,question,answer,correct_attempts,wrong_attempts,value_tuple,value_num,difficulty,topic_cluster,topic_name,human_score
1,1,47825,7532,2017-05-16,Jeopardy,"(5, 2)",INFRASTRUCTURE,"(400,)",False,"The FAA lists airports as small, medium or lar...",hub,0.0,3.0,"(400,)",400.0,easy,5,"Mythology, Literature & Wordplay",0
2,2,10333,2742,1996-07-02,Jeopardy,"(3, 3)",STARS ON CD-ROM,"(300,)",False,Wing Commander III features this Luke Skywalke...,Mark Hamill,1.0,0.0,"(300,)",300.0,easy,12,History & World Leaders,1


In [ ]:
!ollama list


NAME           ID              SIZE     MODIFIED               
gpt-oss:20b    17052f91a42e    13 GB    Less than a second ago    


In [ ]:
import pandas as pd

In [ ]:
final_df= pd.read_csv('adaptive_experiment_final.csv')

In [ ]:
from tqdm import tqdm

tqdm.pandas()  # enables progress_apply

def judge_answer(true_ans, pred_ans):
    prompt = f"""
You are an evaluator.

Determine if the predicted answer matches the true answer semantically.
Minor spelling differences, abbreviations, or equivalent names should count as a match.

True Answer: {true_ans}
Predicted Answer: {pred_ans}

Respond with ONLY "YES" or "NO".
"""

    response = ollama.chat(
        model="gpt-oss:20b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    result = response['message']['content'].strip().upper()

    if "YES" in result:
        return 1
    elif "NO" in result:
        return 0
    return None

In [ ]:
final_df["LLM_judge"] = final_df.progress_apply(
    lambda row: judge_answer(row["true_answer"], row["final_answer"]),
    axis=1
)

100%|██████████| 1000/1000 [15:19<00:00,  1.09it/s]


In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)



In [ ]:
final_df["rouge"] = final_df.apply(
    lambda row: scorer.score(row["true_answer"], row["final_answer"])["rougeL"].fmeasure,
    axis=1
)

In [ ]:
final_df.LLM_judge.mean()

np.float64(0.59)

In [ ]:
final_df.rouge.mean()

np.float64(0.5455447603121516)

In [ ]:
final_df.to_csv('adaptive_experiment_final_real.csv')

In [ ]:
from bert_score import score



In [ ]:
final_df['LLM_judge'].sum()

np.int64(579)

In [ ]:
final_df

,Unnamed: 0.1,Unnamed: 0,ep_num,air_date,round_name,coord,category,value,daily_double,question,...,max_votes,disagreement_level,vote_distribution,total_latency,initial_answers,debated_answers,final_answer,correct,question_id,LLM_judge
0,0,49222,7555,2017-06-16,Double Jeopardy,"(1, 1)",YOU'RE FIRED FROM THE MOVIE!,"(400,)",False,"Spoiler alert! In ""Return of the Jedi"", Luke s...",...,2,high,"{'darth vader': 2, 'vader': 1, 'admiral motti'...",48.411473,"[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...","[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...",darth vader,True,0,1
1,1,47825,7532,2017-05-16,Jeopardy,"(5, 2)",INFRASTRUCTURE,"(400,)",False,"The FAA lists airports as small, medium or lar...",...,2,high,"{'hub': 2, 'medium': 1, 'the faa classifies ai...",49.700103,"[{'model': 'qwen2:7b', 'answer_raw': 'HUB', 'a...","[{'model': 'qwen2:7b', 'answer_raw': 'HUB', 'a...",hub,True,1,1
2,2,10333,2742,1996-07-02,Jeopardy,"(3, 3)",STARS ON CD-ROM,"(300,)",False,Wing Commander III features this Luke Skywalke...,...,1,high,"{'antony felle': 1, 'mark hamill': 1, 'mark ha...",55.676312,"[{'model': 'qwen2:7b', 'answer_raw': 'Antony F...","[{'model': 'qwen2:7b', 'answer_raw': 'Antony F...",mark hamill,True,2,1
3,3,20355,2938,1997-05-14,Jeopardy,"(5, 2)",OUT OF THE MICROWAVE,"(200,)",False,Betty Crocker sells this snack food under the ...,...,2,high,"{'pop secret': 2, 'popcorn': 1, 'no': 1, 'no b...",28.448211,"[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...","[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...",pop secret,False,3,1
4,4,6457,2606,1995-12-25,Jeopardy,"(2, 1)",ETIQUETTE,"(100,)",False,"We're told this ""doesn't grow on trees"", but i...",...,1,high,"{'candle': 1, 'money': 1, 'a wedding ring': 1,...",37.479811,"[{'model': 'qwen2:7b', 'answer_raw': 'Candle',...","[{'model': 'qwen2:7b', 'answer_raw': 'wedding ...",wedding ring,False,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,46580,7511,2017-04-17,Jeopardy,"(5, 5)",GET A LYFT WITH UBER,"(1000,)",False,Airports have been slow to welcome rideshare c...,...,2,high,"{'denver international airport': 1, 'jan 1 201...",36.079641,"[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...","[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...",denver international airport,False,995,0
996,996,18788,2910,1997-04-04,Double Jeopardy,"(2, 3)",MEDICINE,"(600,)",False,Poor bone marrow function is the leading cause...,...,3,high,"{'anemia': 3, 'aplastic anemia': 1, 'diamondbl...",29.960462,"[{'model': 'qwen2:7b', 'answer_raw': 'Anemia',...","[{'model': 'qwen2:7b', 'answer_raw': 'Aplastic...",aplastic anemia,False,996,0
997,997,49635,7562,2017-06-27,Jeopardy,"(2, 5)",OLD SCHOOL,"(1000,)",False,"Founded in 1592, it's also called the Universi...",...,4,medium,"{'trinity college dublin': 4, 'trinity college...",27.511527,"[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...","[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...",trinity college dublin,False,997,1
998,998,34197,4318,2003-05-14,Jeopardy,"(4, 4)",PARIS SITES,"(800,)",False,"Seen here, it's located in the Hôtel des Inval...",...,1,high,{'the final answer the tomb of napoleon bonapa...,29.046867,"[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...","[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...",the tomb of napoleon bonaparte,False,998,1


In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
from tqdm import tqdm

model_columns = ["qwen2", "mistral-nemo", "gemma2", "llama3.1", "deepseek-r1", "phi3"]

for col in model_columns:
    baseline[f"{col}_judge"] = None

for i, row in tqdm(baseline.iterrows(), total=len(baseline)):
    true_ans = row["true_answer"]

    for col in model_columns:
        pred_ans = row[col]
        baseline.loc[i, f"{col}_judge"] = judge_answer(true_ans, pred_ans)

In [ ]:
new_df_data = []

for index, row in final_df.iterrows():
    entry = {
        'question': row['question'],
        'true_answer': row['true_answer']
    }

    # Extract initial answers
    if 'initial_answers' in row and row['initial_answers'] is not None:
        for ans in row['initial_answers']:
            entry[f"{ans['model']}_initial_answer"] = ans['answer_raw']
    else:
        for model_name in TEAM: # Assuming TEAM is defined and contains all model names
            entry[f"{model_name}_initial_answer"] = None

    # Extract debated answers
    if 'debated_answers' in row and row['debated_answers'] is not None:
        for ans in row['debated_answers']:
            entry[f"{ans['model']}_debated_answer"] = ans['answer_raw']
    else:
        for model_name in TEAM:
            entry[f"{model_name}_debated_answer"] = None

    new_df_data.append(entry)

comparison_df = pd.DataFrame(new_df_data)

display(comparison_df.head())

,question,true_answer,qwen2:7b_initial_answer,mistral-nemo:12b_initial_answer,llama3.1:8b_initial_answer,deepseek-r1:8b_initial_answer,phi3:3.8b_initial_answer,qwen2:7b_debated_answer,mistral-nemo:12b_debated_answer,llama3.1:8b_debated_answer,deepseek-r1:8b_debated_answer,phi3:3.8b_debated_answer
0,"Spoiler alert! In ""Return of the Jedi"", Luke s...",Darth Vader,Darth Vader,Vader,Darth Vader,Admiral Motti,Jabba the Hutt burns to death during his trial...,Darth Vader,Vader,Admiral Piett,Darth Vader,Jabba the Hutt is not relevant to this context...
1,"The FAA lists airports as small, medium or lar...",hub,HUB,Medium,HUB.,The FAA classifies airports based on the first...,SMALL,HUB,Medium,SMALL,"The FAA classifies airports as small, medium, ...",llama3.1:8b: HUB
2,Wing Commander III features this Luke Skywalke...,Mark Hamill,Antony Felle,Mark Hamill,Mark Hamill is not in Wing Commander III. The ...,Harrison Ford,Roy Dewayne White,Antony Felle,Mark Hamill,Mark Hamill,Kevin Michael Richardson,Mark Hamill is indeed in Wing Commander III as...
3,Betty Crocker sells this snack food under the ...,Microwave popcorn,Pop Secret,Pop Secret,Popcorn.,No.,"No, Betty Crocker does not sell a product call...",Pop Secret,Popcorn,Betty Crocker sells Pop Secret under license f...,No.,"No, Betty Crocker does not sell a product call..."
4,"We're told this ""doesn't grow on trees"", but i...",money,Candle,Money,The answer is a wedding ring.,gift wrap,"Puzzle candy, like a piñata or balloons might ...",wedding ring,Wedding Ring,Gift wrap.,Puzzle candy,Puzzle candy tied to an artificial tree at a w...


In [ ]:
from tqdm.auto import tqdm
tqdm.pandas() # Ensure tqdm is enabled for pandas

for model_name in TEAM:
    # Judge initial answers
    initial_col = f"{model_name}_initial_answer"
    judge_initial_col = f"{model_name}_initial_judge"
    if initial_col in comparison_df.columns:
        print(f"Judging initial answers for {model_name}...")
        comparison_df[judge_initial_col] = comparison_df.progress_apply(
            lambda row: judge_answer(row['true_answer'], row[initial_col]),
            axis=1
        )

    # Judge debated answers
    debated_col = f"{model_name}_debated_answer"
    judge_debated_col = f"{model_name}_debated_judge"
    if debated_col in comparison_df.columns:
        print(f"Judging debated answers for {model_name}...")
        comparison_df[judge_debated_col] = comparison_df.progress_apply(
            lambda row: judge_answer(row['true_answer'], row[debated_col]),
            axis=1
        )

display(comparison_df.head())

Judging initial answers for qwen2:7b...


  0%|          | 0/1000 [00:00<?, ?it/s]

Judging debated answers for qwen2:7b...


  0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
final_df

,Unnamed: 0.1,Unnamed: 0,ep_num,air_date,round_name,coord,category,value,daily_double,question,...,max_votes,disagreement_level,vote_distribution,total_latency,initial_answers,debated_answers,final_answer,correct,question_id,LLM_judge
0,0,49222,7555,2017-06-16,Double Jeopardy,"(1, 1)",YOU'RE FIRED FROM THE MOVIE!,"(400,)",False,"Spoiler alert! In ""Return of the Jedi"", Luke s...",...,2,high,"{'darth vader': 2, 'vader': 1, 'admiral motti'...",48.411473,"[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...","[{'model': 'qwen2:7b', 'answer_raw': 'Darth Va...",darth vader,True,0,1
1,1,47825,7532,2017-05-16,Jeopardy,"(5, 2)",INFRASTRUCTURE,"(400,)",False,"The FAA lists airports as small, medium or lar...",...,2,high,"{'hub': 2, 'medium': 1, 'the faa classifies ai...",49.700103,"[{'model': 'qwen2:7b', 'answer_raw': 'HUB', 'a...","[{'model': 'qwen2:7b', 'answer_raw': 'HUB', 'a...",hub,True,1,1
2,2,10333,2742,1996-07-02,Jeopardy,"(3, 3)",STARS ON CD-ROM,"(300,)",False,Wing Commander III features this Luke Skywalke...,...,1,high,"{'antony felle': 1, 'mark hamill': 1, 'mark ha...",55.676312,"[{'model': 'qwen2:7b', 'answer_raw': 'Antony F...","[{'model': 'qwen2:7b', 'answer_raw': 'Antony F...",mark hamill,True,2,1
3,3,20355,2938,1997-05-14,Jeopardy,"(5, 2)",OUT OF THE MICROWAVE,"(200,)",False,Betty Crocker sells this snack food under the ...,...,2,high,"{'pop secret': 2, 'popcorn': 1, 'no': 1, 'no b...",28.448211,"[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...","[{'model': 'qwen2:7b', 'answer_raw': 'Pop Secr...",pop secret,False,3,1
4,4,6457,2606,1995-12-25,Jeopardy,"(2, 1)",ETIQUETTE,"(100,)",False,"We're told this ""doesn't grow on trees"", but i...",...,1,high,"{'candle': 1, 'money': 1, 'a wedding ring': 1,...",37.479811,"[{'model': 'qwen2:7b', 'answer_raw': 'Candle',...","[{'model': 'qwen2:7b', 'answer_raw': 'wedding ...",wedding ring,False,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,46580,7511,2017-04-17,Jeopardy,"(5, 5)",GET A LYFT WITH UBER,"(1000,)",False,Airports have been slow to welcome rideshare c...,...,2,high,"{'denver international airport': 1, 'jan 1 201...",36.079641,"[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...","[{'model': 'qwen2:7b', 'answer_raw': 'Denver I...",denver international airport,False,995,0
996,996,18788,2910,1997-04-04,Double Jeopardy,"(2, 3)",MEDICINE,"(600,)",False,Poor bone marrow function is the leading cause...,...,3,high,"{'anemia': 3, 'aplastic anemia': 1, 'diamondbl...",29.960462,"[{'model': 'qwen2:7b', 'answer_raw': 'Anemia',...","[{'model': 'qwen2:7b', 'answer_raw': 'Aplastic...",aplastic anemia,False,996,0
997,997,49635,7562,2017-06-27,Jeopardy,"(2, 5)",OLD SCHOOL,"(1000,)",False,"Founded in 1592, it's also called the Universi...",...,4,medium,"{'trinity college dublin': 4, 'trinity college...",27.511527,"[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...","[{'model': 'qwen2:7b', 'answer_raw': 'Trinity ...",trinity college dublin,False,997,1
998,998,34197,4318,2003-05-14,Jeopardy,"(4, 4)",PARIS SITES,"(800,)",False,"Seen here, it's located in the Hôtel des Inval...",...,1,high,{'the final answer the tomb of napoleon bonapa...,29.046867,"[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...","[{'model': 'qwen2:7b', 'answer_raw': 'The Fina...",the tomb of napoleon bonaparte,False,998,1


In [ ]:
import pandas as pd

In [ ]:
adp_exp=pd.read_csv('adaptive_summary_bertrouge.csv')

In [ ]:
adp_exp

,question,true_answer,ensemble_answer,qwen2:7b,mistral-nemo:12b,llama3.1:8b,deepseek-r1:8b,phi3:3.8b,ensemble_rouge,ensemble_llm_judge,...,mistral-nemo:12b_rouge,llama3.1:8b_rouge,deepseek-r1:8b_rouge,phi3:3.8b_rouge,qwen2:7b_bert_f1,mistral-nemo:12b_bert_f1,llama3.1:8b_bert_f1,deepseek-r1:8b_bert_f1,phi3:3.8b_bert_f1,ensemble_answer_bert_f1
0,"Spoiler alert! In ""Return of the Jedi"", Luke s...",Darth Vader,darth vader,Darth Vader,Vader,Admiral Piett,Darth Vader,Jabba the Hutt is not relevant to this context...,1.000000,1,...,0.666667,0.000000,1.000000,0.025974,1.000000,0.908536,0.860456,1.000000,0.826094,0.899335
1,"The FAA lists airports as small, medium or lar...",hub,hub,HUB,Medium,SMALL,"The FAA classifies airports as small, medium, ...",llama3.1:8b: HUB,1.000000,1,...,0.000000,0.000000,0.000000,0.400000,0.891962,0.848641,0.858538,0.787924,0.814441,1.000000
2,Wing Commander III features this Luke Skywalke...,Mark Hamill,mark hamill,Antony Felle,Mark Hamill,Mark Hamill,Kevin Michael Richardson,Mark Hamill is indeed in Wing Commander III as...,1.000000,1,...,1.000000,1.000000,0.000000,0.080000,0.870064,1.000000,1.000000,0.887869,0.855543,0.868536
3,Betty Crocker sells this snack food under the ...,Microwave popcorn,pop secret,Pop Secret,Popcorn,Betty Crocker sells Pop Secret under license f...,No.,"No, Betty Crocker does not sell a product call...",0.000000,1,...,0.666667,0.000000,0.000000,0.000000,0.818405,0.908240,0.827309,0.865472,0.804098,0.786687
4,"We're told this ""doesn't grow on trees"", but i...",money,wedding ring,wedding ring,Wedding Ring,Gift wrap.,Puzzle candy,Puzzle candy tied to an artificial tree at a w...,0.000000,0,...,0.000000,0.000000,0.000000,0.000000,0.872698,0.855437,0.877796,0.866784,0.798749,0.872699
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Airports have been slow to welcome rideshare c...,Atlanta,new york john f. kennedy international airport...,Denver International Airport,"Jan. 1, 2017",New York John F. Kennedy International Airport...,New York John F. Kennedy International Airport...,"mistral-nemo:12b: Jan. 1, 2neralize that Harts...",0.000000,0,...,0.000000,0.000000,0.000000,0.090909,0.857797,0.863828,0.872250,0.872250,0.815285,0.802750
996,Poor bone marrow function is the leading cause...,Anemia,aplastic anemia,Aplastic Anemia,Aplastic Anemia,Aplastic Anemia,Aplastic Anemia,mistral-nemo:12b: Aplastic Anemia,0.666667,0,...,0.666667,0.666667,0.666667,0.333333,0.930222,0.930222,0.930222,0.930222,0.875633,0.906472
997,"Founded in 1592, it's also called the Universi...",Trinity College,trinity college dublin,Trinity College Dublin,Trinity College Dublin,"Trinity College, Cambridge.",Trinity College Dublin,Trinity College Dublin,0.800000,1,...,0.800000,0.800000,0.800000,0.800000,0.950232,0.950232,0.947530,0.950232,0.950232,0.880175
998,"Seen here, it's located in the Hôtel des Inval...",Napoleon\'s Tomb,the tomb of napoleon bonaparte,The Final Answer: The Tomb of Napoleon Bonaparte,Napoleon,The Dome des Invalides.,The Tomb of Napoleon Bonaparte,The Final Answer: Napoleon's Tomb at Les Inval...,0.250000,1,...,0.500000,0.000000,0.250000,0.081081,0.894668,0.929844,0.868429,0.913767,0.847969,0.881602


In [ ]:
import pandas as pd

# Define the models of interest
models = [
    "qwen2:7b",
    "mistral-nemo:12b",
    "llama3.1:8b",
    "deepseek-r1:8b",
    "phi3:3.8b"
]

# Collect the relevant metrics columns
metrics_cols = []
for model in models:
    # Adding ROUGE and BERT F1 columns based on the existing DataFrame structure
    metrics_cols.append(f"{model}_rouge")
    metrics_cols.append(f"{model}_bert_f1")

# Display the mean scores for a quick comparison
print("Average Scores per Model:")
display(adp_exp[metrics_cols].mean())

# Display the first few rows of these metrics
print("\nDetailed metrics (Head):")
display(adp_exp[metrics_cols].head())

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
from tqdm import tqdm
tqdm.pandas()

# The judge_answer function was defined earlier.
# I will apply it to each of the specified models.
models_to_judge = [ 'phi3:3.8b','llama3.1:8b']

for model in models_to_judge:
    print(f"Judging answers for {model}...")
    ad_judge_col = f"{model}_LLM_judge"
    adp_exp[ad_judge_col] = adp_exp.progress_apply(
        lambda row: judge_answer(row['true_answer'], row[model]),
        axis=1
    )

# Display the average judge scores
print("\nAverage LLM Judge Scores:")
display(adp_exp[[f"{m}_LLM_judge" for m in models_to_judge]].mean())

Judging answers for phi3:3.8b...


100%|██████████| 1000/1000 [18:21<00:00,  1.10s/it]


Judging answers for llama3.1:8b...


100%|██████████| 1000/1000 [16:19<00:00,  1.02it/s]


Average LLM Judge Scores:


,0
phi3:3.8b_LLM_judge,0.311
llama3.1:8b_LLM_judge,0.306


In [ ]:
adp_exp.to_csv('adaptive_summary_bertrouge_final.csv')